# OAuth server

> Let apps such as Claude sign in to a FastHTML app and act as its users

In [ ]:
#| default_exp oauth_server

`OAuthServer` lets apps such as Claude act as the signed-in users of a FastHTML app. An app that calls a protected route without a token, usually an MCP endpoint at `/mcp`, gets a 401 that points to this server. The app registers itself and sends the person to an Allow page. When the person allows it, the app trades a one-time code for an access token and a refresh token. This is OAuth 2.1 as the [MCP authorization spec](https://modelcontextprotocol.io/specification/2025-11-25/basic/authorization) asks for, with the FastHTML app as its own authorization server.

Every sign-in uses PKCE with S256. An access token lasts an hour and works only for the protected route. A refresh token lasts 30 days and works once. Each use returns a new pair of tokens. Using an old refresh token again disconnects the app. The server stores codes and tokens only as SHA-256 hashes.

The app keeps its own sign-in. It tells the server who is signed in, and can supply its own Allow page.

In [ ]:
#| export
import base64, hashlib, json, re, secrets, time
from urllib.parse import urlencode, urlsplit
from fasthtml.common import *
from fasthtml.oauth import _arun, redir_url

In [ ]:
from urllib.parse import parse_qsl
from bs4 import BeautifulSoup
from fastcore.test import *
from fastlite import database
from starlette.testclient import TestClient
from fasthtml.oauth import OAuth

## Storage

Times are whole seconds since the epoch. Codes and tokens are stored as their SHA-256 hashes. A copy of the database holds nothing an app could use:

In [ ]:
#| export
def _now(): return int(time.time())
def _digest(s): return hashlib.sha256(s.encode()).hexdigest()

The three tables hold registered apps, one-time codes, and tokens:

In [ ]:
#| export
class OAuthClient: id:str; name:str; redirect_uris:str; created_at:int
class OAuthCode: hash:str; client_id:str; uid:str; redirect_uri:str; challenge:str; created_at:int
class OAuthToken: hash:str; kind:str; client_id:str; uid:str; created_at:int; revoked:bool

A store holds the tables. `FastliteStore` keeps them in a [fastlite](https://fastlite.answer.ai) database `db`, creating any that don't exist. An app on another database writes a store with the same methods. The server calls each store method through `_arun`. A store's methods can therefore be sync or async.

`take_code` deletes a code and returns it in one statement. `use_refresh` revokes a refresh token and returns it in one statement. Two requests can't redeem the same code or refresh token, even from different processes. `apps` lists the apps holding a live refresh token for a user, newest first:

In [ ]:
#| export
class FastliteStore:
    "The server's tables in fastlite database `db`"
    def __init__(self, db):
        self.db = db
        self.clients = db.create(OAuthClient, name='oauth_client', pk='id', if_not_exists=True)
        self.codes = db.create(OAuthCode, name='oauth_code', pk='hash', if_not_exists=True)
        self.tokens = db.create(OAuthToken, name='oauth_token', pk='hash', if_not_exists=True)

    def add_client(self, client): self.clients.insert(client)
    def client(self, client_id): return self.clients.get(client_id, default=None)
    def clients_since(self, ts): return self.clients.count_where('created_at >= ?', [ts])
    def add_code(self, code): self.codes.insert(code)
    def take_code(self, code_hash, since):
        return first(OAuthCode(**r) for r in self.db.q('DELETE FROM oauth_code WHERE hash=? AND created_at >= ? RETURNING *', [code_hash, since]))
    def add_token(self, token): self.tokens.insert(token)
    def token(self, token_hash): return self.tokens.get(token_hash, default=None)
    def use_refresh(self, token_hash, client_id, since):
        sql = "UPDATE oauth_token SET revoked=1 WHERE hash=? AND kind='refresh' AND client_id=? AND NOT revoked AND created_at >= ? RETURNING *"
        return first(OAuthToken(**r) for r in self.db.q(sql, [token_hash, client_id, since]))
    def revoke(self, token_hash, client_id): self.db.execute('UPDATE oauth_token SET revoked=1 WHERE hash=? AND client_id=?', [token_hash, client_id])
    def disconnect(self, client_id, uid): self.db.execute('UPDATE oauth_token SET revoked=1 WHERE client_id=? AND uid=?', [client_id, uid])
    def apps(self, uid, since):
        sql = """SELECT DISTINCT c.* FROM oauth_client c JOIN oauth_token t ON t.client_id=c.id
            WHERE t.uid=? AND t.kind='refresh' AND NOT t.revoked AND t.created_at >= ? ORDER BY c.created_at DESC"""
        return [OAuthClient(**r) for r in self.db.q(sql, [uid, since])]

A code can be taken once:

In [ ]:
store = FastliteStore(database(':memory:'))
store.add_code(dict(hash='h1', client_id='app1', uid='alice', redirect_uri='https://example.com/cb', challenge='x', created_at=_now()))
test_eq(store.take_code('h1', since=0).uid, 'alice')
test_eq(store.take_code('h1', since=0), None)

## PKCE and return addresses

An app keeps a random verifier and sends its `s256` with the sign-in. Only the app holding the verifier can then trade the code for tokens. `s256` matches the example in [RFC 7636](https://www.rfc-editor.org/rfc/rfc7636#appendix-B):

In [ ]:
#| export
def s256(verifier):
    "PKCE's S256 challenge for `verifier`"
    return base64.urlsafe_b64encode(hashlib.sha256(verifier.encode()).digest()).rstrip(b'=').decode()

In [ ]:
test_eq(s256('dBjftJeZ4CVP-mB92K27uhbUJU1p1r_wW1gFWFOEjXk'), 'E9Melhoa2OwvFrEMTJguCHaoeK1t8URWbuGJSstw-cM')

An app registers the addresses the person may be sent back to. Each must use https, or plain http on a loopback address. A native app such as Claude Code waits on a loopback port that it picks each time. `redirect_ok` checks an address against an app's registrations. A loopback address matches on any port:

In [ ]:
#| export
def loopback(uri):
    "Whether `uri` is plain http on this computer"
    s = urlsplit(uri)
    return s.scheme == 'http' and s.hostname in ('localhost', '127.0.0.1', '::1')

def redirect_ok(client, uri):
    "Whether `uri` is one of `client`'s registered return addresses, with any port on a loopback address"
    regs = json.loads(client.redirect_uris)
    if uri in regs: return True
    s = urlsplit(uri)
    return loopback(uri) and any(loopback(r) and (urlsplit(r).hostname, urlsplit(r).path) == (s.hostname, s.path) for r in regs)

In [ ]:
native = OAuthClient(id='cli', name='A CLI', redirect_uris=json.dumps(['http://localhost:33418/callback']), created_at=0)
assert redirect_ok(native, 'http://localhost:51234/callback')
assert not redirect_ok(native, 'http://localhost:51234/other')
assert not redirect_ok(native, 'https://localhost:51234/callback')

## The Allow page

The Allow page asks the signed-in person whether to let an app act as them. `AllowPage` is its default content: the app's name, where the person goes back to, and a form with Allow and Deny buttons. To use its own layout and wording, such as a list of what a connected app can do, an app passes its own `allow_page` to `OAuthServer`:

In [ ]:
#| export
def AllowPage(req, app_name, where, form):
    "A plain page asking the signed-in person to allow `app_name`"
    return Titled(f'Connect {app_name}?', P(f'{app_name} wants to act as you on this site.'),
        P(f'Only allow this if you just asked {app_name} to connect. Either way, you go back to {where}.'), form)

In [ ]:
AllowPage(None, 'Claude', 'claude.ai', Form(Button('Allow'), Button('Deny')))

(<title>Connect Claude?</title>,
 <main class="container"><h1>Connect Claude?</h1><p>Claude wants to act as you on this site.</p><p>Only allow this if you just asked Claude to connect. Either way, you go back to claude.ai.</p><form enctype="multipart/form-data"><button>Allow</button><button>Deny</button></form></main>)

## The server

`OAuthServer` adds these routes to `app`:

- `/.well-known/oauth-protected-resource`, also with `resource` appended, and `/.well-known/oauth-authorization-server` describe the server to apps.
- `/oauth/register` registers an app.
- `/oauth/authorize` shows the Allow page and takes its form.
- `/oauth/token` gives out tokens.
- `/oauth/revoke` takes a token back.

`current_user` receives the session and tells the server who is signed in. `skip` lists the server's paths and `resource`. Apps call these paths without a session. Add them to the `skip` list of the app's sign-in Beforeware, such as `OAuth`'s.

In [ ]:
#| export
class OAuthServer:
    "An OAuth 2.1 server that lets apps such as Claude act as a signed-in user at `resource`"
    def __init__(self,
        app, # The `FastHTML` app to add the server's routes to
        store, # Where registered apps, codes and tokens live, such as a `FastliteStore`
        current_user, # `current_user(session)` returns the signed-in user's id, or `None`
        resource='/mcp', # The path that access tokens work for
        login_path='/login', # Where a signed-out visitor goes; the way back is in `session['next']`
        allow_page=AllowPage, # `allow_page(req, app_name, where, form)` returns the Allow page's content
        token_user=None, # `token_user(uid, issued_at)` returns who a token acts as, or `None` to refuse it
        base_url=None, # The site's public origin, such as `https://example.com`; worked out from each request if `None`
        code_ttl=600, # Seconds a one-time code lasts
        access_ttl=3600, # Seconds an access token lasts
        refresh_ttl=30*24*3600, # Seconds a refresh token lasts
        regs_per_day=200, # App registrations allowed in 24 hours
    ):
        store_attr()
        meta = '/.well-known/oauth-protected-resource'
        self.skip = [meta, meta+resource, '/.well-known/oauth-authorization-server', '/oauth/register', '/oauth/authorize', '/oauth/token', '/oauth/revoke', resource]
        app.get(meta, name='oauth_resource')(self.resource_metadata)
        app.get(meta+resource, name='oauth_resource_path')(self.resource_metadata)
        app.get('/.well-known/oauth-authorization-server', name='oauth_server')(self.server_metadata)
        app.post('/oauth/register', name='oauth_register')(self.register)
        app.get('/oauth/authorize', name='oauth_authorize')(self.authorize)
        app.post('/oauth/authorize', name='oauth_allow')(self.allow)
        app.post('/oauth/token', name='oauth_token')(self.token)
        app.post('/oauth/revoke', name='oauth_revoke')(self.revoke)

`url` gives the public address of a path. It uses `base_url` when set. Otherwise `redir_url` works it out from the request. A protected route answers a request without a valid token with `unauthorized`. Its `WWW-Authenticate` header points to the resource metadata, where an app learns how to sign in:

In [ ]:
#| export
@patch
def url(self:OAuthServer, req, path=''):
    "The public address of `path` on this site"
    return self.base_url.rstrip('/')+path if self.base_url else redir_url(req, path)

@patch
def unauthorized(self:OAuthServer, req):
    "A 401 that points an app to the resource metadata"
    meta = f'Bearer resource_metadata="{self.url(req, "/.well-known/oauth-protected-resource")}"'
    if 'authorization' in req.headers: meta += ', error="invalid_token"'
    return Response('unauthorized', status_code=401, headers={'WWW-Authenticate': meta})

The resource metadata (RFC 9728) names this site as the server that signs people in for `resource`. The server metadata (RFC 8414) lists the server's endpoints and what it supports:

In [ ]:
#| export
@patch
def resource_metadata(self:OAuthServer, req):
    "RFC 9728: `resource` is protected, and this site signs people in for it"
    return dict(resource=self.url(req, self.resource), authorization_servers=[self.url(req)], bearer_methods_supported=['header'])

@patch
def server_metadata(self:OAuthServer, req):
    "RFC 8414: where an app registers, sends people to allow it, and gets and revokes tokens"
    b = self.url(req)
    return dict(issuer=b, authorization_endpoint=f'{b}/oauth/authorize', token_endpoint=f'{b}/oauth/token', registration_endpoint=f'{b}/oauth/register',
        revocation_endpoint=f'{b}/oauth/revoke', response_types_supported=['code'], grant_types_supported=['authorization_code', 'refresh_token'],
        code_challenge_methods_supported=['S256'], token_endpoint_auth_methods_supported=['none'], revocation_endpoint_auth_methods_supported=['none'])

`register` implements RFC 7591 registration. An app sends its name and return addresses. It gets back a client id with no secret. Registration needs no sign-in. `regs_per_day` caps how many apps register in 24 hours. `_err` is an OAuth error reply:

In [ ]:
#| export
def _err(code, why='', status=400): return JSONResponse(dict(error=code, error_description=why), status_code=status, headers={'Cache-Control': 'no-store'})

@patch
async def register(self:OAuthServer, req):
    "RFC 7591: register an app from its metadata, and reply with its client id"
    try: d = await req.json()
    except ValueError: return _err('invalid_client_metadata', 'Send the client metadata as JSON.')
    uris = d.get('redirect_uris') if isinstance(d, dict) else None
    if not (isinstance(uris, list) and 0 < len(uris) <= 10 and all(isinstance(u, str) and len(u) <= 500 and (u.startswith('https://') or loopback(u)) for u in uris)):
        return _err('invalid_redirect_uri', 'Redirect addresses must use https, or http on a loopback address.')
    if await _arun(self.store.clients_since(_now()-24*3600)) >= self.regs_per_day: return _err('temporarily_unavailable', 'Too many new apps today. Try again tomorrow.', 429)
    c = dict(id=secrets.token_urlsafe(18), name=' '.join(str(d.get('client_name') or 'An app').split())[:80], redirect_uris=json.dumps(uris), created_at=_now())
    await _arun(self.store.add_client(c))
    return JSONResponse(dict(client_id=c['id'], client_id_issued_at=c['created_at'], client_name=c['name'], redirect_uris=uris,
        token_endpoint_auth_method='none', grant_types=['authorization_code', 'refresh_token'], response_types=['code']), status_code=201)

`authorize` checks an app's sign-in link before showing anything. An unknown app or an unregistered return address gets an error page and no redirect. A redirect to that address can't be trusted. Other mistakes go back to the app as OAuth errors, through `_back`. A signed-out visitor goes to `login_path`, with the link saved in `session['next']` for the way back. Every page that `authorize` and `allow` show sends `X-Frame-Options: DENY`. Another site then can't frame the page and trick a click on Allow:

In [ ]:
#| export
def _back(uri, state, **q):
    "Send the person back to the app at `uri` with `q` and the app's `state`"
    return RedirectResponse(uri + ('&' if '?' in uri else '?') + urlencode(q | ({'state': state} if state else {})), status_code=303)

@patch
def _page(self:OAuthServer, *c): return HttpHeader('X-Frame-Options', 'DENY'), *c

@patch
async def authorize(self:OAuthServer, req, session, response_type:str='', client_id:str='', redirect_uri:str='', code_challenge:str='',
    code_challenge_method:str='', state:str='', resource:str=''):
    "The Allow page: check the app's link, send a signed-out visitor to sign in, and ask the person to allow the app"
    c = await _arun(self.store.client(client_id)) if client_id else None
    if not c or not redirect_ok(c, redirect_uri): return self._page(Titled('Connect an app', P('This link to connect an app is broken or out of date. Start again from the app.')))
    if response_type != 'code': return _back(redirect_uri, state, error='unsupported_response_type')
    if code_challenge_method != 'S256' or not re.fullmatch(r'[A-Za-z0-9._~-]{43,128}', code_challenge):
        return _back(redirect_uri, state, error='invalid_request', error_description='PKCE with S256 is required.')
    if resource and resource.rstrip('/') != self.url(req, self.resource):
        return _back(redirect_uri, state, error='invalid_target', error_description=f'Tokens here are only for {self.resource}.')
    if not await _arun(self.current_user(session)):
        session['next'] = f'{req.url.path}?{req.url.query}'
        return RedirectResponse(self.login_path, status_code=303)
    session['oauth_csrf'] = csrf = secrets.token_urlsafe(16)
    keep = dict(client_id=c.id, redirect_uri=redirect_uri, code_challenge=code_challenge, state=state, csrf=csrf)
    form = Form(*[Hidden(v, name=k) for k,v in keep.items()], Button('Allow'), Button('Deny', name='deny', value='1'), method='post', action='/oauth/authorize')
    where = 'an app on this computer' if loopback(redirect_uri) else urlsplit(redirect_uri).hostname
    return self._page(await _arun(self.allow_page(req, c.name, where, form)))

`allow` takes the Allow page's form. It checks the form's nonce against the one `authorize` saved in the session. Allowing sends the app a one-time code tied to the app, the user, the return address and the PKCE challenge:

In [ ]:
#| export
@patch
async def allow(self:OAuthServer, req, session, client_id:str='', redirect_uri:str='', code_challenge:str='', state:str='', csrf:str='', deny:str=''):
    "Allow or deny the app; allowing sends it a one-time code"
    c = await _arun(self.store.client(client_id)) if client_id else None
    uid, expected = await _arun(self.current_user(session)), session.pop('oauth_csrf', '')
    if not (c and uid and redirect_ok(c, redirect_uri) and expected and secrets.compare_digest(csrf, expected)):
        return self._page(Titled('Connect an app', P('This page has expired. Start again from the app.')))
    if deny: return _back(redirect_uri, state, error='access_denied')
    code = secrets.token_urlsafe(32)
    await _arun(self.store.add_code(dict(hash=_digest(code), client_id=c.id, uid=uid, redirect_uri=redirect_uri, challenge=code_challenge, created_at=_now())))
    return _back(redirect_uri, state, code=code)

`token` trades a one-time code and its PKCE verifier for tokens, or a refresh token for a new pair. A refresh token used a second time disconnects the app. `_live` applies `token_user`. It can refuse a token, for example one issued before the user signed out of all devices:

In [ ]:
#| export
@patch
async def _live(self:OAuthServer, t):
    "Who token row `t` acts as, or `None` when `token_user` refuses it"
    return await _arun(self.token_user(t.uid, t.created_at)) if self.token_user else t.uid

@patch
async def _new_tokens(self:OAuthServer, client_id, uid):
    "A new access token and refresh token for app `client_id` and user `uid`, as RFC 6749's token reply"
    access, refresh = secrets.token_urlsafe(32), secrets.token_urlsafe(32)
    for tok,kind in ((access,'access'), (refresh,'refresh')):
        await _arun(self.store.add_token(dict(hash=_digest(tok), kind=kind, client_id=client_id, uid=uid, created_at=_now(), revoked=False)))
    return JSONResponse(dict(access_token=access, token_type='Bearer', expires_in=self.access_ttl, refresh_token=refresh), headers={'Cache-Control': 'no-store'})

@patch
async def token(self:OAuthServer, req, grant_type:str='', client_id:str='', code:str='', code_verifier:str='', redirect_uri:str='', refresh_token:str='', resource:str=''):
    "Trade a one-time code and its PKCE verifier, or a refresh token, for new tokens"
    if resource and resource.rstrip('/') != self.url(req, self.resource): return _err('invalid_target', f'Tokens here are only for {self.resource}.')
    if grant_type == 'authorization_code':
        k = await _arun(self.store.take_code(_digest(code), _now()-self.code_ttl))
        if not k or k.client_id != client_id or k.redirect_uri != redirect_uri: return _err('invalid_grant', 'The code is wrong, used or out of date.')
        if not secrets.compare_digest(s256(code_verifier), k.challenge): return _err('invalid_grant', 'The PKCE verifier does not match.')
        return await self._new_tokens(client_id, k.uid)
    if grant_type != 'refresh_token': return _err('unsupported_grant_type')
    t = await _arun(self.store.use_refresh(_digest(refresh_token), client_id, _now()-self.refresh_ttl))
    if not t:
        old = await _arun(self.store.token(_digest(refresh_token)))
        if old and old.kind == 'refresh' and old.client_id == client_id and old.revoked: await _arun(self.store.disconnect(client_id, old.uid))
        return _err('invalid_grant', 'The refresh token is unknown, used or out of date.')
    if not await self._live(t): return _err('invalid_grant', 'This sign-in no longer works. Connect the app again.')
    return await self._new_tokens(client_id, t.uid)

`revoke` implements RFC 7009. An app gives up a token through it. Revoking an unknown token isn't an error:

In [ ]:
#| export
@patch
async def revoke(self:OAuthServer, token:str='', client_id:str=''):
    "RFC 7009: app `client_id` gives up `token`"
    await _arun(self.store.revoke(_digest(token), client_id))
    return Response(status_code=200)

A protected route calls `user` to find who a request's access token acts as. `apps` lists the apps a user has connected. `disconnect` ends every token one app holds for that user. An account page can list the apps with a Disconnect button for each:

In [ ]:
#| export
@patch
async def user(self:OAuthServer, req):
    "Who the request's access token acts as, or `None`"
    hdr = req.headers.get('authorization', '')
    t = await _arun(self.store.token(_digest(hdr[7:]))) if hdr.startswith('Bearer ') else None
    if not t or t.kind != 'access' or t.revoked or t.created_at < _now()-self.access_ttl: return None
    return await self._live(t)

@patch
async def apps(self:OAuthServer, uid, since=0):
    "The apps `uid` has connected, newest first: those holding a live refresh token issued after `since`"
    return await _arun(self.store.apps(uid, max(since, _now()-self.refresh_ttl)))

@patch
async def disconnect(self:OAuthServer, client_id, uid):
    "End every token that app `client_id` holds for `uid`"
    await _arun(self.store.disconnect(client_id, uid))

## Connecting an app

This app's `/login` signs the visitor in as `alice` and returns to the page saved in `session['next']`:

In [ ]:
app = FastHTML(secret_key='soopersecret')
srv = OAuthServer(app, FastliteStore(database(':memory:')), current_user=lambda session: session.get('auth'))

@app.get('/login')
def login(session):
    session['auth'] = 'alice'
    return RedirectResponse(session.pop('next', '/'), status_code=303)

Its route at `/mcp` stands in for an MCP server. It answers with the user that a token acts as, and with `unauthorized` when there is no valid token:

In [ ]:
@app.post('/mcp')
async def mcp(req):
    if not (uid := await srv.user(req)): return srv.unauthorized(req)
    return dict(user=uid)

cli = TestClient(app)

Without a token, `/mcp` answers 401 and names the resource metadata:

In [ ]:
r = cli.post('/mcp')
test_eq(r.status_code, 401)
r.headers['www-authenticate']

'Bearer resource_metadata="https://testserver/.well-known/oauth-protected-resource"'

The resource metadata names the server that signs people in. The server metadata lists its endpoints:

In [ ]:
meta = cli.get('/.well-known/oauth-protected-resource/mcp').json()
test_eq(cli.get('/.well-known/oauth-authorization-server').json()['token_endpoint'], 'https://testserver/oauth/token')
meta

{'resource': 'https://testserver/mcp',
 'authorization_servers': ['https://testserver'],
 'bearer_methods_supported': ['header']}

Claude registers with the address claude.ai sends people back to:

In [ ]:
cb = 'https://claude.ai/api/mcp/auth_callback'
r = cli.post('/oauth/register', json=dict(client_name='Claude', redirect_uris=[cb]))
test_eq(r.status_code, 201)
claude = r.json()
claude

{'client_id': 'zwyZosZqEGqMhunRQeXUKa_h',
 'client_id_issued_at': 1794969913,
 'client_name': 'Claude',
 'redirect_uris': ['https://claude.ai/api/mcp/auth_callback'],
 'token_endpoint_auth_method': 'none',
 'grant_types': ['authorization_code', 'refresh_token'],
 'response_types': ['code']}

An address that is neither https nor loopback is refused:

In [ ]:
test_eq(cli.post('/oauth/register', json=dict(redirect_uris=['http://example.com/cb'])).json()['error'], 'invalid_redirect_uri')

Claude's sign-in link carries its client id, its return address, a PKCE challenge, its `state` and the resource it wants tokens for:

In [ ]:
verifier = secrets.token_urlsafe(48)
q = dict(response_type='code', client_id=claude['client_id'], redirect_uri=cb, code_challenge=s256(verifier), code_challenge_method='S256', state='xyz', resource=meta['resource'])
auth_url = f'/oauth/authorize?{urlencode(q)}'

A signed-out visitor is sent to sign in, then back to the same link:

In [ ]:
test_eq(cli.get(auth_url, follow_redirects=False).headers['location'], '/login')
test_eq(cli.get('/login', follow_redirects=False).headers['location'], auth_url)

Signed in, the person sees the Allow page. No other site may show it in a frame:

In [ ]:
r = cli.get(auth_url)
test_eq(r.headers['x-frame-options'], 'DENY')
assert 'Connect Claude?' in r.text and 'claude.ai' in r.text

`allow_claude` presses Allow by posting the page's form. It returns the query that Claude gets back:

In [ ]:
def allow_claude():
    "Press Allow on Claude's Allow page, and return the query Claude gets back"
    page = cli.get(auth_url).text
    fields = {i['name']: i['value'] for i in BeautifulSoup(page, 'html.parser').select('form input')}
    r = cli.post('/oauth/authorize', data=fields, follow_redirects=False)
    return dict(parse_qsl(urlsplit(r.headers['location']).query))

back = allow_claude()
test_eq(back['state'], 'xyz')
back.keys()

dict_keys(['code', 'state'])

A link naming an unknown app gets the error page, with no redirect. A sign-in without S256 goes back to the app with an OAuth error:

In [ ]:
r = cli.get(f'/oauth/authorize?{urlencode(q | dict(client_id="nope"))}', follow_redirects=False)
assert r.status_code == 200 and 'broken or out of date' in r.text
r = cli.get(f'/oauth/authorize?{urlencode(q | dict(code_challenge_method="plain"))}', follow_redirects=False)
assert r.headers['location'].startswith(f'{cb}?error=invalid_request')

Claude trades the code and its verifier for tokens. The code works once:

In [ ]:
tok_req = dict(grant_type='authorization_code', client_id=claude['client_id'], redirect_uri=cb, code_verifier=verifier, resource=meta['resource'])
toks = cli.post('/oauth/token', data=tok_req | dict(code=back['code'])).json()
test_eq(cli.post('/oauth/token', data=tok_req | dict(code=back['code'])).json()['error'], 'invalid_grant')
toks.keys()

dict_keys(['access_token', 'token_type', 'expires_in', 'refresh_token'])

A code redeemed with the wrong verifier gives no tokens:

In [ ]:
code = allow_claude()['code']
test_eq(cli.post('/oauth/token', data=tok_req | dict(code=code, code_verifier='x'*43)).json()['error_description'], 'The PKCE verifier does not match.')

With the access token, `/mcp` runs as `alice`:

In [ ]:
def call_mcp(tok): return cli.post('/mcp', headers={'Authorization': f'Bearer {tok}'})
test_eq(call_mcp(toks['access_token']).json(), {'user': 'alice'})

A refresh token gives one new pair. Using it again disconnects Claude. The new access token stops working too:

In [ ]:
ref = dict(grant_type='refresh_token', client_id=claude['client_id'], refresh_token=toks['refresh_token'])
new = cli.post('/oauth/token', data=ref).json()
test_eq(call_mcp(new['access_token']).status_code, 200)
test_eq(cli.post('/oauth/token', data=ref).json()['error'], 'invalid_grant')
test_eq(call_mcp(new['access_token']).status_code, 401)

After Claude connects again, `apps` lists it. `disconnect` ends its tokens:

In [ ]:
toks = cli.post('/oauth/token', data=tok_req | dict(code=allow_claude()['code'])).json()
test_eq([a.name for a in await srv.apps('alice')], ['Claude'])
await srv.disconnect(claude['client_id'], 'alice')
test_eq(await srv.apps('alice'), [])
test_eq(call_mcp(toks['access_token']).status_code, 401)

`token_user` can refuse tokens issued before a time, such as when the user last signed out of all devices:

In [ ]:
toks = cli.post('/oauth/token', data=tok_req | dict(code=allow_claude()['code'])).json()
signed_out = _now()+1
srv.token_user = lambda uid, issued_at: uid if issued_at >= signed_out else None
test_eq(call_mcp(toks['access_token']).status_code, 401)
srv.token_user = None

## Signing in with `OAuth`

When FastHTML's `OAuth` handles sign-in, pass the server's paths in its `skip`. After sign-in, `OAuth` returns to the Allow page saved in `session['next']`. Here `StubClient` stands in for a provider such as Google. It signs everyone in as `bob`:

In [ ]:
class StubClient:
    "A provider that signs everyone in as `bob`"
    id_key = 'sub'
    async def retr_info_async(self, code, redirect_uri): return dict(sub='bob')

class Auth(OAuth):
    def get_auth(self, info, ident, session, state): return RedirectResponse('/', status_code=303)

app2 = FastHTML(secret_key='soopersecret')
srv2 = OAuthServer(app2, FastliteStore(database(':memory:')), current_user=lambda session: session.get('auth'))
oauth = Auth(app2, StubClient(), skip=[*srv2.skip, '/redirect', '/error', '/login'])
cli2 = TestClient(app2)

A signed-out visitor goes to `/login`. After the provider sends them back to `/redirect`, `OAuth` returns them to the Allow page:

In [ ]:
client2 = cli2.post('/oauth/register', json=dict(client_name='Claude', redirect_uris=[cb])).json()
link = f"/oauth/authorize?{urlencode(q | dict(client_id=client2['client_id']))}"
test_eq(cli2.get(link, follow_redirects=False).headers['location'], '/login')
test_eq(cli2.get('/redirect?code=abc', follow_redirects=False).headers['location'], link)
assert 'Connect Claude?' in cli2.get(link).text

# Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()